# Notebook 05 — Hierarchical Models: Partial Pooling

## Background

Suppose you're analyzing sales data across 50 stores, or test scores across 30 schools, or clinical trial outcomes across 12 hospitals. You have two obvious options:

1. **Complete pooling**: treat all groups as identical, fit one global model. Fast, but ignores genuine group differences.
2. **No pooling**: fit each group completely independently. Respects group differences, but groups with few observations get noisy, unreliable estimates.

Hierarchical models offer a third option: **partial pooling**. Each group gets its own estimate, but those estimates are *shrunk toward a global mean* based on how much data the group has. Small groups borrow strength from the global pattern; large groups are more self-sufficient. The amount of shrinkage is learned from the data, not set manually.

This is mathematically equivalent to putting a prior on the group-level parameters where the prior is itself learned from the data. That's the "hierarchical" part — there are parameters at multiple levels, with higher-level parameters governing the distribution of lower-level ones.

## What You'll Learn

- Complete pooling vs. no pooling vs. partial pooling — side-by-side comparison
- Building hierarchical models in PyMC: varying intercepts, varying slopes
- The Bayesian shrinkage effect: small groups get more shrinkage
- Neal's funnel and the non-centered parameterization (the most important PyMC skill)
- Cross-validation intuition for why partial pooling wins

## Why This Matters

Hierarchical models are one of the most practically impactful tools in Bayesian analysis. Almost any dataset with a grouping structure benefits from them: A/B tests across user segments, medical outcomes across clinics, sensor readings across devices. The non-centered parameterization covered here is also the key to sampling hierarchical models without divergences — it's the single most common PyMC fix practitioners need.

In [ ]:
import pymc as pm
import arviz as az
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

print(f'PyMC {pm.__version__}  |  ArviZ {az.__version__}')

## Part 1 — The Three Approaches: Setup

We'll use a classic setup: estimating the conversion rate for 20 ad campaigns. Some campaigns have lots of data, some have very few impressions. This contrast makes the partial pooling benefit obvious.

In [ ]:
np.random.seed(42)

n_campaigns = 20

# True conversion rates: drawn from a population distribution
# In reality we never observe these -- the hierarchical model estimates them
true_mu    = 0.12   # true population mean conversion rate
true_sigma = 0.05   # true between-campaign variability

true_rates = np.clip(
    np.random.normal(true_mu, true_sigma, n_campaigns), 0.01, 0.99
)

# Vary campaign sizes dramatically: some have 20 impressions, some have 2000
campaign_sizes = np.array([
    20, 30, 25, 500, 800, 1000, 20, 40, 200, 1500,
    15, 25, 400, 600, 30, 18, 900, 1200, 20, 35
])

# Observed conversions
conversions = np.random.binomial(campaign_sizes, true_rates)
observed_rates = conversions / campaign_sizes

# Sort by campaign size for visualization
sort_idx = np.argsort(campaign_sizes)

print(f'Campaigns: {n_campaigns}')
print(f'True population: mu={true_mu}, sigma={true_sigma}')
print(f'Campaign sizes: {campaign_sizes.min()} -- {campaign_sizes.max()}')
print(f'Observed rates: {observed_rates.min():.3f} -- {observed_rates.max():.3f}')
print(f'True rates:     {true_rates.min():.3f} -- {true_rates.max():.3f}')

In [ ]:
# ── Model 1: Complete Pooling ─────────────────────────────────────────────────
# Single theta for all campaigns -- ignores group differences

with pm.Model() as complete_pool:
    theta = pm.Beta('theta', alpha=1, beta=1)  # single shared rate
    y = pm.Binomial('y', n=campaign_sizes, p=theta, observed=conversions)
    trace_cp = pm.sample(draws=1000, tune=500, chains=2,
                         random_seed=42, progressbar=False)

cp_mean = float(trace_cp.posterior['theta'].values.mean())
print(f'Complete pooling estimate: {cp_mean:.4f} (same for all campaigns)')
print(f'True population mean:      {true_mu:.4f}')

In [ ]:
# ── Model 2: No Pooling ───────────────────────────────────────────────────────
# Separate independent Beta posterior for each campaign

# Analytical no-pooling posteriors: Beta(alpha + successes, beta + failures)
# With uniform Beta(1,1) prior: posterior = Beta(1+k, 1+n-k)
alpha_prior, beta_prior = 1, 1
np_means = (alpha_prior + conversions) / (alpha_prior + beta_prior + campaign_sizes)
np_ci_lo = np.array([stats.beta.ppf(0.03, alpha_prior + k, beta_prior + n - k)
                     for k, n in zip(conversions, campaign_sizes)])
np_ci_hi = np.array([stats.beta.ppf(0.97, alpha_prior + k, beta_prior + n - k)
                     for k, n in zip(conversions, campaign_sizes)])

print('No-pooling (independent Beta posteriors):')
print(f'  CI width for small campaigns (n~20):   {(np_ci_hi - np_ci_lo)[campaign_sizes < 40].mean():.3f}')
print(f'  CI width for large campaigns (n>500):  {(np_ci_hi - np_ci_lo)[campaign_sizes > 500].mean():.3f}')

In [ ]:
# ── Model 3: Hierarchical (Partial Pooling) ───────────────────────────────────
# theta_i ~ Beta(alpha, beta) where alpha, beta learned from data
# Equivalently: logit(theta_i) ~ Normal(mu_logit, sigma_logit)
# Second form is easier to sample -- use it

with pm.Model() as hierarchical_model:
    # Hyperpriors: population-level distribution of conversion rates
    mu_logit    = pm.Normal('mu_logit', mu=0, sigma=2)      # population mean (logit scale)
    sigma_logit = pm.HalfNormal('sigma_logit', sigma=1)     # between-campaign variability
    
    # ── Non-centered parameterization ────────────────────────────────────────
    # CENTERED (what NOT to do):
    #   theta_raw_logit ~ Normal(mu_logit, sigma_logit)  <- divergences in practice
    # NON-CENTERED (correct approach):
    #   z_i ~ Normal(0, 1)   <- easy for NUTS
    #   theta_logit_i = mu_logit + sigma_logit * z_i  <- deterministic transformation
    
    z = pm.Normal('z', mu=0, sigma=1, shape=n_campaigns)  # standard normal offsets
    theta_logit = pm.Deterministic(
        'theta_logit', mu_logit + sigma_logit * z
    )
    theta = pm.Deterministic(
        'theta', pm.math.sigmoid(theta_logit)
    )
    
    y = pm.Binomial('y', n=campaign_sizes, p=theta, observed=conversions)
    
    trace_hier = pm.sample(
        draws=2000, tune=1000, chains=4,
        target_accept=0.9,
        random_seed=42,
        progressbar=True
    )

n_div = int(trace_hier.sample_stats['diverging'].values.sum())
rhat_max = float(az.summary(trace_hier, var_names=['mu_logit', 'sigma_logit'])['r_hat'].max())
print(f'Divergences: {n_div}  |  Max R-hat: {rhat_max:.4f}')

In [ ]:
# Extract hierarchical estimates
theta_samples = trace_hier.posterior['theta'].values  # shape: (chains, draws, campaigns)
hier_means = theta_samples.mean(axis=(0, 1))
hier_ci_lo = np.percentile(theta_samples, 3, axis=(0, 1))
hier_ci_hi = np.percentile(theta_samples, 97, axis=(0, 1))

# Compare all three approaches
fig, ax = plt.subplots(figsize=(14, 7))

x = np.arange(n_campaigns)
idx = sort_idx  # sorted by campaign size

# True rates
ax.scatter(x, true_rates[idx], marker='*', s=120, color='black', zorder=6, label='True rate', alpha=0.9)

# Complete pooling (same for all)
ax.axhline(cp_mean, color='gray', linestyle='-', lw=1.5, alpha=0.7, label=f'Complete pooling: {cp_mean:.3f}')

# No pooling (observed rates)
ax.scatter(x - 0.15, observed_rates[idx], color='darkorange', s=40, alpha=0.8, zorder=4, label='No pooling (observed)')
ax.vlines(x - 0.15, np_ci_lo[idx], np_ci_hi[idx], color='darkorange', alpha=0.4, lw=1)

# Hierarchical partial pooling
ax.scatter(x + 0.15, hier_means[idx], color='steelblue', s=40, alpha=0.9, zorder=5, label='Partial pooling (hierarchical)')
ax.vlines(x + 0.15, hier_ci_lo[idx], hier_ci_hi[idx], color='steelblue', alpha=0.4, lw=1)

# Mark small vs large campaigns
small_mask = campaign_sizes[idx] < 50
ax.axvspan(-0.5, np.where(small_mask)[0].max() + 0.5, alpha=0.05, color='red', label='Small campaigns (n<50)')

ax.set_xticks(x)
ax.set_xticklabels([f'n={campaign_sizes[idx[i]]}' for i in range(n_campaigns)],
                    rotation=70, fontsize=7)
ax.set_ylabel('Conversion Rate')
ax.set_title('Partial Pooling vs. Complete Pooling vs. No Pooling\n'
             'Campaigns sorted by size (left=small, right=large)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Quantify the shrinkage effect
shrinkage = 1 - np.abs(hier_means - cp_mean) / np.abs(observed_rates - cp_mean + 1e-10)
shrinkage = np.clip(shrinkage, 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Shrinkage vs campaign size
ax = axes[0]
scatter = ax.scatter(campaign_sizes, shrinkage, c=campaign_sizes, cmap='viridis',
                     s=60, alpha=0.8)
plt.colorbar(scatter, ax=ax, label='Campaign size (n)')
ax.set_xlabel('Campaign Size (n)')
ax.set_ylabel('Shrinkage toward population mean')
ax.set_title('Bayesian Shrinkage\nSmall n -> strong shrinkage toward global mean')

# Error comparison
ax = axes[1]
mse_no_pool = np.mean((observed_rates - true_rates) ** 2)
mse_hier    = np.mean((hier_means - true_rates) ** 2)
mse_cp      = np.mean((np.full(n_campaigns, cp_mean) - true_rates) ** 2)

models = ['Complete\nPooling', 'No Pooling\n(observed)', 'Hierarchical\n(partial pool)']
mses   = [mse_cp, mse_no_pool, mse_hier]
colors = ['gray', 'darkorange', 'steelblue']
bars = ax.bar(models, mses, color=colors, alpha=0.8, edgecolor='white')
ax.set_ylabel('MSE vs True Rates')
ax.set_title('Mean Squared Error\nHierarchical wins on small-n groups')
for bar, mse in zip(bars, mses):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0001,
            f'{mse:.5f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print(f'MSE comparison:')
print(f'  Complete pooling: {mse_cp:.5f}')
print(f'  No pooling:       {mse_no_pool:.5f}')
print(f'  Hierarchical:     {mse_hier:.5f}')
print(f'\nHierarchical MSE reduction vs no-pooling: {(mse_no_pool - mse_hier) / mse_no_pool:.1%}')

## Part 2 — The Non-Centered Parameterization

This is the single most important PyMC skill for hierarchical models. The centered parameterization creates Neal's funnel — a geometry where divergences are common. The non-centered version flattens the geometry and eliminates them.

**Centered**: `theta_i ~ Normal(mu, sigma)`
- PyMC samples `theta_i` and `(mu, sigma)` jointly
- When `sigma` is small (groups are similar), `theta_i` is tightly constrained by `(mu, sigma)` — the posterior is a narrow funnel
- NUTS must use tiny step sizes in the narrow part → divergences

**Non-centered**: `z_i ~ Normal(0, 1); theta_i = mu + sigma * z_i`
- PyMC samples `z_i` and `(mu, sigma)` jointly
- `z_i` and `(mu, sigma)` are nearly independent in the prior — flat geometry
- NUTS works efficiently regardless of `sigma`

In [ ]:
# Demonstrate centered vs non-centered on a simple hierarchical model
# Small sigma regime -> centered model diverges, non-centered does not

np.random.seed(42)
n_groups = 8
n_per_group = 5
true_group_mu = 3.0
true_group_sigma = 0.1  # small sigma = tight funnel

group_means_true = np.random.normal(true_group_mu, true_group_sigma, n_groups)
y_data = np.array([
    np.random.normal(mu, 0.5, n_per_group) for mu in group_means_true
]).flatten()
group_idx = np.repeat(np.arange(n_groups), n_per_group)

# Centered parameterization
with pm.Model() as centered_model:
    mu    = pm.Normal('mu', mu=0, sigma=5)
    sigma = pm.HalfNormal('sigma', sigma=2)
    # theta_i directly from Normal(mu, sigma) -- the problematic form
    theta = pm.Normal('theta', mu=mu, sigma=sigma, shape=n_groups)
    y_obs = pm.Normal('y_obs', mu=theta[group_idx], sigma=0.5, observed=y_data)
    trace_centered = pm.sample(draws=500, tune=500, chains=2,
                                random_seed=42, progressbar=False)

n_div_centered = int(trace_centered.sample_stats['diverging'].values.sum())

# Non-centered parameterization
with pm.Model() as noncentered_model:
    mu    = pm.Normal('mu', mu=0, sigma=5)
    sigma = pm.HalfNormal('sigma', sigma=2)
    # z_i ~ Normal(0, 1), then theta_i = mu + sigma * z_i
    z     = pm.Normal('z', mu=0, sigma=1, shape=n_groups)
    theta = pm.Deterministic('theta', mu + sigma * z)
    y_obs = pm.Normal('y_obs', mu=theta[group_idx], sigma=0.5, observed=y_data)
    trace_nc = pm.sample(draws=500, tune=500, chains=2,
                          random_seed=42, progressbar=False)

n_div_nc = int(trace_nc.sample_stats['diverging'].values.sum())

print('Centered parameterization:')
print(f'  Divergences: {n_div_centered}')
print(az.summary(trace_centered, var_names=['mu', 'sigma'])['r_hat'].to_string())
print()
print('Non-centered parameterization:')
print(f'  Divergences: {n_div_nc}')
print(az.summary(trace_nc, var_names=['mu', 'sigma'])['r_hat'].to_string())

In [ ]:
# Visualize: centered creates funnel, non-centered is flat
# Plot (sigma, theta[0]) joint for both parameterizations

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (trace, label, color) in zip(axes, [
    (trace_centered, 'Centered -- divergences show up as spikes', 'steelblue'),
    (trace_nc,       'Non-centered -- no divergences', 'seagreen'),
]):
    sigma_samp = trace.posterior['sigma'].values.flatten()
    theta0_samp = trace.posterior['theta'].values.reshape(-1, n_groups)[:, 0]
    diverging = trace.sample_stats['diverging'].values.flatten().astype(bool)
    
    ax.scatter(sigma_samp[~diverging], theta0_samp[~diverging],
               alpha=0.15, s=8, color=color)
    if diverging.sum() > 0:
        ax.scatter(sigma_samp[diverging], theta0_samp[diverging],
                   alpha=0.9, s=40, color='red', marker='x',
                   label=f'Divergences: {diverging.sum()}')
    
    ax.set_xlabel('sigma (group SD)')
    ax.set_ylabel('theta[0] (group mean)')
    ax.set_title(label)
    if diverging.sum() > 0:
        ax.legend()

plt.suptitle('Centered vs. Non-Centered Parameterization\n'
             'Red x = divergent transitions (biased samples)', fontsize=12)
plt.tight_layout()
plt.show()

## Part 3 — Varying Intercepts and Varying Slopes

The partial-pooling idea extends naturally to regression: instead of just shrinking group means, we can shrink group-specific regression coefficients. A *varying intercepts* model lets each group have its own baseline; a *varying slopes* model lets each group have its own response to a predictor.

In [ ]:
# Scenario: price elasticity of demand across 10 product categories
# Each category has its own intercept (baseline demand) and slope (price sensitivity)
# We want to estimate both while sharing information across categories

np.random.seed(42)
n_categories = 10
n_obs_per_cat = 15

# True population parameters
true_mu_alpha  = 5.0   # mean log-demand at reference price
true_sig_alpha = 0.8   # between-category baseline variation
true_mu_beta   = -0.6  # mean price elasticity (negative: higher price -> lower demand)
true_sig_beta  = 0.3   # between-category elasticity variation

true_alphas = np.random.normal(true_mu_alpha, true_sig_alpha, n_categories)
true_betas  = np.random.normal(true_mu_beta,  true_sig_beta,  n_categories)

# Simulate data
price_list, demand_list, cat_idx_list = [], [], []
for c in range(n_categories):
    price = np.random.uniform(0.5, 3.0, n_obs_per_cat)
    log_demand = true_alphas[c] + true_betas[c] * price + np.random.normal(0, 0.3, n_obs_per_cat)
    price_list.append(price)
    demand_list.append(log_demand)
    cat_idx_list.append(np.full(n_obs_per_cat, c))

prices    = np.concatenate(price_list)
demands   = np.concatenate(demand_list)
cat_idx   = np.concatenate(cat_idx_list).astype(int)

print(f'Dataset: {len(prices)} observations across {n_categories} categories')
print(f'True elasticities: {true_betas.min():.2f} to {true_betas.max():.2f}')
print(f'True baselines:    {true_alphas.min():.2f} to {true_alphas.max():.2f}')

In [ ]:
# Varying intercepts AND slopes -- non-centered throughout
with pm.Model() as varying_slopes_model:
    # Population-level priors
    mu_alpha  = pm.Normal('mu_alpha',  mu=0, sigma=5)        # avg baseline
    mu_beta   = pm.Normal('mu_beta',   mu=0, sigma=2)        # avg elasticity
    sig_alpha = pm.HalfNormal('sig_alpha', sigma=2)          # between-cat baseline SD
    sig_beta  = pm.HalfNormal('sig_beta',  sigma=1)          # between-cat elasticity SD
    
    # Non-centered: raw standard Normal offsets per category
    z_alpha = pm.Normal('z_alpha', mu=0, sigma=1, shape=n_categories)
    z_beta  = pm.Normal('z_beta',  mu=0, sigma=1, shape=n_categories)
    
    # Deterministic: actual intercepts and slopes per category
    alpha = pm.Deterministic('alpha', mu_alpha + sig_alpha * z_alpha)
    beta  = pm.Deterministic('beta',  mu_beta  + sig_beta  * z_beta)
    
    # Likelihood
    mu_pred = alpha[cat_idx] + beta[cat_idx] * prices
    sigma_obs = pm.HalfNormal('sigma_obs', sigma=1)
    y_obs = pm.Normal('y_obs', mu=mu_pred, sigma=sigma_obs, observed=demands)
    
    trace_vs = pm.sample(
        draws=2000, tune=1000, chains=4,
        target_accept=0.9,
        random_seed=42,
        progressbar=True
    )

n_div = int(trace_vs.sample_stats['diverging'].values.sum())
rhat_summary = az.summary(trace_vs, var_names=['mu_alpha', 'mu_beta', 'sig_alpha', 'sig_beta'])
print(f'Divergences: {n_div}  |  Max R-hat: {rhat_summary["r_hat"].max():.4f}')
print(rhat_summary.to_string())

In [ ]:
# Recover true category-level parameters
alpha_samp = trace_vs.posterior['alpha'].values.reshape(-1, n_categories)
beta_samp  = trace_vs.posterior['beta'].values.reshape(-1, n_categories)

alpha_means = alpha_samp.mean(axis=0)
beta_means  = beta_samp.mean(axis=0)
alpha_ci    = np.percentile(alpha_samp, [3, 97], axis=0)
beta_ci     = np.percentile(beta_samp,  [3, 97], axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (param_means, param_ci, true_vals, param_name, mu_true, sig_true) in zip(axes, [
    (alpha_means, alpha_ci, true_alphas, 'alpha (baseline)', true_mu_alpha, true_sig_alpha),
    (beta_means,  beta_ci,  true_betas,  'beta (elasticity)', true_mu_beta, true_sig_beta),
]):
    cats = np.arange(n_categories)
    ax.scatter(cats, true_vals, marker='*', s=120, color='black', zorder=6, label='True')
    ax.scatter(cats, param_means, color='steelblue', s=50, zorder=5, label='Posterior mean')
    ax.vlines(cats, param_ci[0], param_ci[1], color='steelblue', alpha=0.5, lw=1.5)
    ax.axhline(mu_true, color='red', linestyle='--', lw=1.5, label=f'True population mean: {mu_true}')
    mu_est_mean = float(trace_vs.posterior[f'mu_{param_name[0]}'].values.mean())
    ax.axhline(mu_est_mean, color='steelblue', linestyle='--', lw=1,
               label=f'Estimated population mean: {mu_est_mean:.2f}')
    ax.set_xticks(cats); ax.set_xticklabels([f'Cat {i}' for i in cats], rotation=45)
    ax.set_ylabel(param_name); ax.set_title(f'Hierarchical Recovery: {param_name}\n94% CI shown')
    ax.legend(fontsize=8)

plt.suptitle('Varying Intercepts + Slopes: Category-Level Recovery', fontsize=12)
plt.tight_layout()
plt.show()

print('Correlation: posterior mean vs true values')
print(f'  Alpha: {np.corrcoef(alpha_means, true_alphas)[0,1]:.3f}')
print(f'  Beta:  {np.corrcoef(beta_means,  true_betas)[0,1]:.3f}')

In [ ]:
# Visualize per-category demand curves with posterior uncertainty
fig, axes = plt.subplots(2, 5, figsize=(16, 7), sharey=True)

price_range = np.linspace(0.5, 3.0, 100)

for c, ax in enumerate(axes.flatten()):
    # Posterior predictive curves for this category
    mu_curves = alpha_samp[:, c, None] + beta_samp[:, c, None] * price_range[None, :]
    
    ax.fill_between(price_range,
                    np.percentile(mu_curves, 3, axis=0),
                    np.percentile(mu_curves, 97, axis=0),
                    alpha=0.25, color='steelblue')
    ax.plot(price_range, mu_curves.mean(axis=0), 'steelblue', lw=1.5)
    ax.plot(price_range, true_alphas[c] + true_betas[c] * price_range, 'r--', lw=1.5, alpha=0.8)
    ax.scatter(prices[cat_idx == c], demands[cat_idx == c],
               s=15, alpha=0.5, color='steelblue')
    ax.set_title(f'Cat {c}\nb={beta_means[c]:.2f}', fontsize=9)
    ax.set_xlabel('Price', fontsize=8)
    if c % 5 == 0:
        ax.set_ylabel('log(demand)', fontsize=8)

plt.suptitle('Per-Category Demand Curves\nBlue=posterior, Red dashed=true, Shading=94% CI', fontsize=12)
plt.tight_layout()
plt.show()

## Part 4 — Why Partial Pooling Beats the Alternatives

The formal argument for hierarchical models comes from Stein's paradox (1956) and empirical Bayes theory: when estimating multiple parameters simultaneously from noisy data, shrinking toward a common mean *always* reduces total squared error compared to independent estimates — as long as there are 3 or more groups.

Intuitively: the noisy estimate for a small group isn't just noisy — it's systematically pulled toward extreme values by chance. Shrinkage corrects this. And the global mean estimate is much more reliable than any individual group estimate, making it a useful regularizer.

The Bayesian hierarchical model formalizes this: the group-level prior (with parameters learned from data) is the shrinkage target, and the posterior automatically balances how much to shrink each group based on its sample size.

In [ ]:
# Summary: parameterization cheatsheet
print('Hierarchical Model Cheatsheet')
print('=' * 60)
print()
print('CENTERED (avoid for hierarchical models):')
print('  theta_i ~ Normal(mu, sigma)')
print('  Problem: funnel geometry -> divergences when sigma is small')
print()
print('NON-CENTERED (always use for hierarchical models):')
print('  z_i ~ Normal(0, 1)            <- standard normal offset')
print('  theta_i = mu + sigma * z_i    <- deterministic transformation')
print('  Works because z_i and (mu, sigma) are nearly independent')
print()
print('When to use centered:')
print('  - Large n_per_group and small n_groups: data overwhelms the prior')
print('    and centered can occasionally sample better in that regime')
print('  - But non-centered is a safe default')
print()
print('PyMC template:')
code = '''
with pm.Model():
    # Hyperpriors
    mu    = pm.Normal('mu',    mu=0, sigma=5)
    sigma = pm.HalfNormal('sigma', sigma=2)
    
    # Non-centered group offsets
    z     = pm.Normal('z', mu=0, sigma=1, shape=n_groups)
    theta = pm.Deterministic('theta', mu + sigma * z)
    
    # Likelihood
    y_obs = pm.Normal('y_obs', mu=theta[group_idx], sigma=sigma_obs, observed=y)
    
    trace = pm.sample(draws=2000, tune=1000, target_accept=0.9)
'''
print(code)

## Summary

| Approach | Group estimates | Between-group info | Small-n behavior |
|----------|----------------|-------------------|------------------|
| Complete pooling | All identical | Maximal | Biased |
| No pooling | Fully independent | None | Noisy/extreme |
| Partial pooling | Shrunk toward mean | Shared via prior | Stabilized |

**Key rules**:
- Always use **non-centered parameterization** for hierarchical models in PyMC
  - `z ~ Normal(0, 1); theta = mu + sigma * z` instead of `theta ~ Normal(mu, sigma)`
- Set `target_accept=0.9` (or higher) for hierarchical models
- Shrinkage is strongest for small groups — this is a feature, not a bug
- The population-level parameters `mu` and `sigma` are themselves inferred: they describe the between-group distribution

**Next**: Notebook 06 — Bayesian Regression & GLMs. Linear, logistic, Poisson, and negative binomial models in PyMC, with posterior predictive checks and interpretation of results.